<a href="https://colab.research.google.com/github/simonmi2/MechInterp/blob/main/qwen_LayerProbing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
!pip install -q git+https://github.com/anthropics/jacobian-lens.git
!pip install -q bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00


In [4]:
!pip install -q transformers accelerate

from huggingface_hub import login
login()

In [5]:
!pip install -q git+https://github.com/anthropics/jacobian-lens.git
!pip install -q bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch, jlens
from jlens import ActivationRecorder
import matplotlib.pyplot as plt

model_name = "meta-llama/Llama-3.3-70B-Instruct"
bnb_config = BitsAndBytesConfig(load_in_4bit=True)
tokenizer = AutoTokenizer.from_pretrained(model_name)
hf_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
hf_model.eval()
lm = jlens.from_hf(hf_model, tokenizer)
print("Model loaded")
print(f"Layers: {lm.n_layers}")

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.3-70B-Instruct.
403 Client Error. (Request ID: Root=1-6a540167-29a01f4b5968c29e5186defd;767db8f5-5bbb-4319-bc14-3d8d1ed6337d)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.3-70B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.3-70B-Instruct is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Llama-3.3-70B-Instruct to ask for access.

In [7]:
from huggingface_hub import list_repo_files

files = list(list_repo_files("neuronpedia/jacobian-lens"))
qwen3_32b_files = [f for f in files if "qwen3-32b" in f.lower()]
for f in qwen3_32b_files:
    print(f)

qwen3-32b/.DS_Store
qwen3-32b/jlens/.DS_Store
qwen3-32b/jlens/Salesforce-wikitext/Qwen3-32B_convergence.csv
qwen3-32b/jlens/Salesforce-wikitext/Qwen3-32B_jacobian_lens.pt
qwen3-32b/jlens/Salesforce-wikitext/config.yaml


In [8]:
from huggingface_hub import login
login()


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch, jlens
from jlens import ActivationRecorder

model_name = "Qwen/Qwen3-32B"
bnb_config = BitsAndBytesConfig(load_in_4bit=True)
tokenizer = AutoTokenizer.from_pretrained(model_name)
hf_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
hf_model.eval()
lm = jlens.from_hf(hf_model, tokenizer)
print("Model loaded")
print(f"Layers: {lm.n_layers}")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/58.3k [00:00<?, ?B/s]

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

In [ ]:
lens = jlens.JacobianLens.from_pretrained(
    "neuronpedia/jacobian-lens",
    filename="qwen3-32b/jlens/Salesforce-wikitext/Qwen3-32B_jacobian_lens.pt"
)
print("Lens loaded")
print(f"Source layers: {lens.source_layers[:5]}...{lens.source_layers[-5:]}")